# 课程 05 - 主动式 RAG


## 设置

本笔记本演示了使用 Microsoft Agent Framework 的 Agentic RAG（检索增强生成）模式。

**先决条件：**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — 你的 Azure AI 搜索服务端点
- `AZURE_SEARCH_API_KEY` — 你的 Azure AI 搜索 API 密钥
- 通过环境变量配置的 Azure OpenAI 部署
- 已通过 Azure CLI 认证（`az login`）


In [ ]:
%pip install agent-framework python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.openai import OpenAIChatCompletionClient

dotenv.load_dotenv()

endpoint = os.getenv("LLM_BASE_URL")
deployment_name = os.getenv("LLM_MODEL")

missing = [k for k, v in {
    "LLM_BASE_URL": endpoint,
    "LLM_MODEL": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
import os
# Create the the chat model provider client
client = OpenAIChatCompletionClient(
    model=os.environ["LLM_MODEL"],
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"],
)

## 什么是 Agentic RAG？

传统的 RAG 遵循固定流程：先检索文档，然后生成回答。**Agentic RAG** 更进一步，赋予代理自主权以决定<strong>何时</strong>以及<strong>如何</strong>检索信息。

使用 Agentic RAG，代理可以：
- <strong>决定</strong> 在回答问题前是否需要检索
- <strong>选择</strong> 查询哪个数据源或工具
- <strong>评估</strong> 检索到的结果，并在第一次尝试不足时执行后续检索
- <strong>整合</strong> 多个检索步骤的信息，形成连贯回答

这使得代理相比静态的先检索后生成流程更加灵活和精准。


## 创建搜索工具

在 Agentic RAG 中，外部数据源被包装为代理可以按需调用的 <strong>工具</strong>。这让代理把检索当成它可以执行的另一个操作，而不是一个强制步骤。

下面我们定义一个旅游知识库，并将其公开为代理可以调用的工具，以查询目的地信息。


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {'Barcelona': "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.", 'Tokyo': "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.", 'Paris': "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.", 'Cape Town': "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150."}

@tool(approval_mode='never_require')
def search_travel_knowledge(query: Annotated[str, 'The search query about a travel destination']) -> str:
    """搜索旅游知识库以获取目的地信息。"""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any((word in info.lower() for word in query.lower().split())):
            results.append(f'**{destination}**: {info}')
    return '\n\n'.join(results) if results else 'No matching destinations found in the knowledge base.'

## 构建 RAG 代理

现在我们创建一个指示为<strong>总是在回答前检索信息</strong>的代理。该代理使用 `search_travel_knowledge` 工具将其回答基于知识库，而不是依赖自身的训练数据。


In [ ]:
agent = client.as_agent(tools=[search_travel_knowledge], name='TravelRAGAgent', instructions='你是一位知识渊博的旅行顾问。在回答有关目的地的问题之前：\n1. 务必先搜索旅行知识库\n2. 根据检索到的信息作答\n3. 如果知识库中没有相关信息，请明确说明\n4. 提供具体细节，例如费用、最佳季节和亮点。')
response = await agent.run('我对参观建筑出色的地方很感兴趣。你有什么目的地推荐吗？')
print(response)

## 迭代检索 — 制作者-审核者模式

Agentic RAG 的一个关键优势是<strong>迭代检索</strong>。代理可以执行多轮搜索，以验证、完善或扩展其初始发现 —— 类似于“制作者-审核者”的工作流程：

1. <strong>制作者步骤</strong>：代理检索初始信息并起草回答。
2. <strong>审核者步骤</strong>：代理进行额外检索以核实细节或填补空白。

下面，代理被问及一个需要比较多个目的地的问题，促使它进行多次搜索。


In [ ]:
checker_agent = client.as_agent(tools=[search_travel_knowledge], name='TravelRAGCheckerAgent', instructions='你是一位一丝不苟的旅行顾问，会反复核查推荐内容。\n回答旅行问题时：\n1. 先搜索相关目的地\n2. 对于找到的每个目的地，再次使用目的地名称搜索以获取完整详情\n3. 使用已验证的信息对比各个选项\n4. 给出最终推荐，包括具体费用、最佳旅行时间和亮点\n5. 如果有任何细节看起来不完整，在回复前再搜索一次以确认。')
response = await checker_agent.run('我的预算是每天175美元，想在四月出行。哪些目的地符合我的预算和时间安排？')
print(response)

## 总结

在本课中，您学习了如何使用 Microsoft Agent Framework 构建一个 **Agentic RAG** 系统：

- **Agentic RAG** 允许代理自主决定何时检索信息，使检索变得动态而非固定。
- <strong>作为数据源的工具</strong>：外部知识库（如 Azure AI Search）被封装为代理可以调用的工具。
- <strong>迭代检索</strong>：制作者-审核者模式使代理能够执行多轮检索——搜索、验证和细化——然后再生成最终答案。

在生产环境中，您会用真实的 Azure AI Search 索引替换内存中的 `TRAVEL_KNOWLEDGE_BASE`，以处理大规模的旅行文档检索。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
